# Step 03: Data Cleaning & Processing Pipeline

## Overview
This notebook cleans, standardizes, and normalizes all 5 raw datasets, producing clean production files in `data/processed/`:
1. `employee_attrition_processed.csv`
2. `engagement_processed.csv`
3. `occupation_master.csv`
4. `essential_skills_processed.csv`
5. `software_skills_processed.csv`

### Key Cleaning Operations:
- Whitespace stripping on all string/text columns.
- Removal of redundant constant/zero-variance features (`EmployeeCount`, `StandardHours`, `Over18`).
- Standardization of date formats to ISO 8601 (`YYYY-MM-DD`).
- Skill & tool name normalization (e.g. "AWS" / "Amazon Web Services" -> "Amazon Web Services (AWS)").


In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)


---
## 1. Clean `employee_attrition.csv`


In [2]:
attr_df = pd.read_csv(os.path.join(RAW_DIR, "employee_attrition.csv"))

# Strip whitespace from string columns
str_cols = attr_df.select_dtypes(include='object').columns
for col in str_cols:
    attr_df[col] = attr_df[col].astype(str).str.strip()

# Drop zero-variance / constant columns
constant_cols = [c for c in attr_df.columns if attr_df[c].nunique() <= 1]
print(f"Dropping constant columns: {constant_cols}")
attr_df.drop(columns=constant_cols, inplace=True)

# Save processed file
out_path_attr = os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv")
attr_df.to_csv(out_path_attr, index=False)
print(f"Saved: {out_path_attr} | Shape: {attr_df.shape}")


Dropping constant columns: ['EmployeeCount', 'Over18', 'StandardHours']
Saved: ..\data\processed\employee_attrition_processed.csv | Shape: (1470, 32)


---
## 2. Clean `hr_performance_engagement.csv`


In [3]:
perf_df = pd.read_csv(os.path.join(RAW_DIR, "hr_performance_engagement.csv"))

# Strip whitespace from string columns
str_cols = perf_df.select_dtypes(include='object').columns
for col in str_cols:
    perf_df[col] = perf_df[col].astype(str).str.strip()

# Standardize date columns
date_cols = ['StartDate', 'Survey Date', 'DOB', 'Training Date']
for dcol in date_cols:
    if dcol in perf_df.columns:
        perf_df[dcol] = pd.to_datetime(perf_df[dcol], errors='coerce').dt.strftime('%Y-%m-%d')

# Save processed file
out_path_perf = os.path.join(PROCESSED_DIR, "engagement_processed.csv")
perf_df.to_csv(out_path_perf, index=False)
print(f"Saved: {out_path_perf} | Shape: {perf_df.shape}")


C:\Users\komal\AppData\Local\Temp\ipykernel_11820\3939962827.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  perf_df[dcol] = pd.to_datetime(perf_df[dcol], errors='coerce').dt.strftime('%Y-%m-%d')
C:\Users\komal\AppData\Local\Temp\ipykernel_11820\3939962827.py:12: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  perf_df[dcol] = pd.to_datetime(perf_df[dcol], errors='coerce').dt.strftime('%Y-%m-%d')
C:\Users\komal\AppData\Local\Temp\ipykernel_11820\3939962827.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  perf_df[dcol] = pd.to_datetime(perf_df[dcol], errors='coerce').dt.strftime('%Y-%m-%d')


Saved: ..\data\processed\engagement_processed.csv | Shape: (2845, 28)


---
## 3. Clean `occupation_data.csv`


In [4]:
occ_df = pd.read_csv(os.path.join(RAW_DIR, "occupation_data.csv"))

# Strip whitespace
for col in occ_df.columns:
    occ_df[col] = occ_df[col].astype(str).str.strip()

out_path_occ = os.path.join(PROCESSED_DIR, "occupation_master.csv")
occ_df.to_csv(out_path_occ, index=False)
print(f"Saved: {out_path_occ} | Shape: {occ_df.shape}")


Saved: ..\data\processed\occupation_master.csv | Shape: (1016, 3)


---
## 4. Clean `essential_skills.csv`


In [5]:
ess_df = pd.read_csv(os.path.join(RAW_DIR, "essential_skills.csv"))

# Strip string columns
for col in ess_df.select_dtypes(include='object').columns:
    ess_df[col] = ess_df[col].astype(str).str.strip()

# Filter to Importance scale (IM) and drop unneeded columns
ess_clean = ess_df[ess_df['Scale ID'] == 'IM'].copy()
ess_clean.drop(columns=['Not Relevant', 'Recommend Suppress'], errors='ignore', inplace=True)

out_path_ess = os.path.join(PROCESSED_DIR, "essential_skills_processed.csv")
ess_clean.to_csv(out_path_ess, index=False)
print(f"Saved: {out_path_ess} | Shape: {ess_clean.shape}")


Saved: ..\data\processed\essential_skills_processed.csv | Shape: (9100, 13)


---
## 5. Clean `software_skills.csv` & Skill Normalization


In [6]:
soft_df = pd.read_csv(os.path.join(RAW_DIR, "software_skills.csv"))

# Strip string columns
for col in soft_df.select_dtypes(include='object').columns:
    soft_df[col] = soft_df[col].astype(str).str.strip()

# Normalization dictionary for common tech/software variations
skill_norm_map = {
    "AWS": "Amazon Web Services (AWS)",
    "Amazon Web Services": "Amazon Web Services (AWS)",
    "Amazon Web Services AWS": "Amazon Web Services (AWS)",
    "MS Excel": "Microsoft Excel",
    "Excel": "Microsoft Excel",
    "Microsoft Excel": "Microsoft Excel",
    "Python programming language": "Python",
    "Python": "Python",
    "Javascript": "JavaScript",
    "JavaScript": "JavaScript",
    "SQL Server": "Microsoft SQL Server",
    "Microsoft SQL Server": "Microsoft SQL Server"
}

soft_df['Normalized_Tool_Name'] = soft_df['Workplace Example'].replace(skill_norm_map)

out_path_soft = os.path.join(PROCESSED_DIR, "software_skills_processed.csv")
soft_df.to_csv(out_path_soft, index=False)
print(f"Saved: {out_path_soft} | Shape: {soft_df.shape}")


Saved: ..\data\processed\software_skills_processed.csv | Shape: (31821, 8)


---
## Summary of Processed Outputs
All 5 processed CSV files successfully generated in `data/processed/`:
- `employee_attrition_processed.csv` (Constant columns removed: `EmployeeCount`, `StandardHours`, `Over18`)
- `engagement_processed.csv` (Dates converted to ISO `YYYY-MM-DD`)
- `occupation_master.csv` (Cleaned occupation reference table)
- `essential_skills_processed.csv` (Filtered to Importance scale `IM`)
- `software_skills_processed.csv` (Added `Normalized_Tool_Name` mapped column)
